# Mesh Generation in Finite Element Analysis

## Introduction: Why Mesh Quality Matters

The mesh is the foundation of any FEA simulation. A good mesh ensures:
- **Accuracy**: Captures geometry and solution gradients properly
- **Stability**: Avoids numerical issues in the solver
- **Efficiency**: Uses computational resources wisely

This notebook covers both structured and unstructured mesh generation techniques.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from scipy.spatial import Delaunay
from scipy.sparse import lil_matrix
import matplotlib.tri as mtri

# Set up nice plotting parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

## 1. Structured vs Unstructured Meshes

### Structured Meshes
Regular grid patterns, easy to generate, good for simple geometries.

### Unstructured Meshes
Flexible element shapes, excellent for complex geometries.

In [ ]:
def create_structured_mesh(nx, ny, width=1.0, height=1.0):
    """
    Create a structured rectangular mesh.
    """
    x = np.linspace(0, width, nx)
    y = np.linspace(0, height, ny)
    X, Y = np.meshgrid(x, y)
    
    nodes = np.column_stack([X.ravel(), Y.ravel()])
    
    # Create elements (quadrilaterals split into triangles)
    elements = []
    for j in range(ny-1):
        for i in range(nx-1):
            n0 = j*nx + i
            n1 = j*nx + i + 1
            n2 = (j+1)*nx + i + 1
            n3 = (j+1)*nx + i
            
            # Split quad into two triangles
            elements.append([n0, n1, n2])
            elements.append([n0, n2, n3])
    
    return nodes, np.array(elements)

def create_unstructured_mesh(n_points=100, domain_radius=1.0):
    """
    Create an unstructured mesh using random points and Delaunay triangulation.
    """
    np.random.seed(42)
    
    # Generate random points in circular domain
    r = np.sqrt(np.random.rand(n_points)) * domain_radius
    theta = np.random.rand(n_points) * 2 * np.pi
    points = np.column_stack([r * np.cos(theta), r * np.sin(theta)])
    
    # Add boundary points
    n_boundary = 30
    boundary_angles = np.linspace(0, 2*np.pi, n_boundary, endpoint=False)
    boundary_points = np.column_stack([
        domain_radius * np.cos(boundary_angles),
        domain_radius * np.sin(boundary_angles)
    ])
    
    nodes = np.vstack([points, boundary_points])
    
    # Delaunay triangulation
    tri = Delaunay(nodes)
    elements = tri.simplices
    
    return nodes, elements

# Create both types of meshes
nodes_struct, elem_struct = create_structured_mesh(6, 6)
nodes_unstruct, elem_unstruct = create_unstructured_mesh(80)

print(f"Structured mesh: {len(nodes_struct)} nodes, {len(elem_struct)} elements")
print(f"Unstructured mesh: {len(nodes_unstruct)} nodes, {len(elem_unstruct)} elements")

In [ ]:
# Visualize both meshes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Structured mesh
ax1.triplot(nodes_struct[:, 0], nodes_struct[:, 1], elem_struct, 'b-', linewidth=0.5)
ax1.plot(nodes_struct[:, 0], nodes_struct[:, 1], 'ro', markersize=3)
ax1.set_title('Structured Mesh')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Unstructured mesh
ax2.triplot(nodes_unstruct[:, 0], nodes_unstruct[:, 1], elem_unstruct, 'g-', linewidth=0.5)
ax2.plot(nodes_unstruct[:, 0], nodes_unstruct[:, 1], 'ro', markersize=3)
ax2.set_title('Unstructured Mesh')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Mesh Quality Metrics

Good mesh quality is crucial for accurate FEA results. Key metrics:
- **Aspect Ratio**: Ratio of longest to shortest edge
- **Minimum Angle**: Smallest angle in triangular elements
- **Area Ratio**: For quadrilateral elements

In [ ]:
def triangle_quality(nodes, elements):
    """
    Compute quality metrics for triangular elements.
    """
    n_elem = len(elements)
    aspect_ratios = np.zeros(n_elem)
    min_angles = np.zeros(n_elem)
    
    for i, elem in enumerate(elements):
        # Get triangle vertices
        v0, v1, v2 = nodes[elem]
        
        # Edge lengths
        e0 = np.linalg.norm(v1 - v2)
        e1 = np.linalg.norm(v0 - v2)
        e2 = np.linalg.norm(v0 - v1)
        
        # Aspect ratio (longest/shortest edge)
        max_edge = max(e0, e1, e2)
        min_edge = min(e0, e1, e2)
        aspect_ratios[i] = max_edge / min_edge
        
        # Minimum angle using law of cosines
        angle0 = np.arccos(np.clip((e1**2 + e2**2 - e0**2) / (2*e1*e2), -1, 1))
        angle1 = np.arccos(np.clip((e0**2 + e2**2 - e1**2) / (2*e0*e2), -1, 1))
        angle2 = np.arccos(np.clip((e0**2 + e1**2 - e2**2) / (2*e0*e1), -1, 1))
        min_angles[i] = min(angle0, angle1, angle2) * 180 / np.pi
    
    return aspect_ratios, min_angles

# Analyze mesh quality
ar_struct, min_ang_struct = triangle_quality(nodes_struct, elem_struct)
ar_unstruct, min_ang_unstruct = triangle_quality(nodes_unstruct, elem_unstruct)

print("Structured Mesh Quality:")
print(f"  Aspect ratio - Mean: {np.mean(ar_struct):.2f}, Max: {np.max(ar_struct):.2f}")
print(f"  Minimum angle - Mean: {np.mean(min_ang_struct):.1f}°, Min: {np.min(min_ang_struct):.1f}°")

print("\nUnstructured Mesh Quality:")
print(f"  Aspect ratio - Mean: {np.mean(ar_unstruct):.2f}, Max: {np.max(ar_unstruct):.2f}")
print(f"  Minimum angle - Mean: {np.mean(min_ang_unstruct):.1f}°, Min: {np.min(min_ang_unstruct):.1f}°")

In [ ]:
# Visualize quality metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Aspect ratio distribution
ax1.hist(ar_struct, bins=20, alpha=0.7, label='Structured', density=True)
ax1.hist(ar_unstruct, bins=20, alpha=0.7, label='Unstructured', density=True)
ax1.set_xlabel('Aspect Ratio')
ax1.set_ylabel('Density')
ax1.set_title('Aspect Ratio Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Minimum angle distribution
ax2.hist(min_ang_struct, bins=20, alpha=0.7, label='Structured', density=True)
ax2.hist(min_ang_unstruct, bins=20, alpha=0.7, label='Unstructured', density=True)
ax2.set_xlabel('Minimum Angle (degrees)')
ax2.set_ylabel('Density')
ax2.set_title('Minimum Angle Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Quality Guidelines:")
print("- Aspect ratio < 5: Good")
print("- Minimum angle > 30°: Good")
print("- Minimum angle > 45°: Excellent")
print("\nStructured meshes typically have better quality metrics.")

## 3. Delaunay Triangulation

Delaunay triangulation maximizes minimum angles, making it ideal for FEA meshes.

In [ ]:
def demonstrate_delaunay():
    """
    Demonstrate Delaunay triangulation properties.
    """
    # Create test points
    np.random.seed(123)
    points = np.random.rand(20, 2)
    
    # Delaunay triangulation
    tri = Delaunay(points)
    
    # Create circumcircles for visualization
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot triangulation
    ax.triplot(points[:, 0], points[:, 1], tri.simplices, 'b-', linewidth=1)
    ax.plot(points[:, 0], points[:, 1], 'ro', markersize=6)
    
    # Plot circumcircles for a few triangles
    for i, simplex in enumerate(tri.simplices[:5]):  # First 5 triangles only
        triangle = points[simplex]
        
        # Calculate circumcenter
        A = np.array([
            [2*(triangle[1,0]-triangle[0,0]), 2*(triangle[1,1]-triangle[0,1])],
            [2*(triangle[2,0]-triangle[0,0]), 2*(triangle[2,1]-triangle[0,1])]
        ])
        
        B = np.array([
            triangle[1,0]**2 + triangle[1,1]**2 - triangle[0,0]**2 - triangle[0,1]**2,
            triangle[2,0]**2 + triangle[2,1]**2 - triangle[0,0]**2 - triangle[0,1]**2
        ])
        
        if np.linalg.det(A) != 0:
            center = np.linalg.solve(A, B)
            radius = np.linalg.norm(triangle[0] - center)
            
            circle = plt.Circle(center, radius, fill=False, 
                              edgecolor='red', alpha=0.5, linewidth=1)
            ax.add_patch(circle)
    
    ax.set_title('Delaunay Triangulation with Circumcircles')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    plt.show()
    
    print("Delaunay Property:")
    print("No circumcircle contains any other points in its interior.")
    print("This maximizes minimum angles, avoiding skinny triangles.")

demonstrate_delaunay()

## 4. Mesh Refinement

Adaptive refinement concentrates elements where needed most.

In [ ]:
def adaptive_refinement(nodes, elements, refinement_criteria=None, max_levels=2):
    """
    Simple adaptive refinement demonstration.
    """
    if max_levels == 0:
        return nodes, elements
    
    # For demonstration, refine elements in center region
    refined_nodes = list(nodes)
    refined_elements = []
    
    for elem in elements:
        triangle = nodes[elem]
        centroid = np.mean(triangle, axis=0)
        
        # Refine if near center
        if np.linalg.norm(centroid - np.array([0.5, 0.5])) < 0.3:
            # Split triangle into 4 smaller triangles
            mid01 = (nodes[elem[0]] + nodes[elem[1]]) / 2
            mid12 = (nodes[elem[1]] + nodes[elem[2]]) / 2
            mid20 = (nodes[elem[2]] + nodes[elem[0]]) / 2
            
            # Add new nodes
            n_mid01 = len(refined_nodes)
            n_mid12 = len(refined_nodes) + 1
            n_mid20 = len(refined_nodes) + 2
            
            refined_nodes.extend([mid01, mid12, mid20])
            
            # Create 4 new triangles
            refined_elements.extend([
                [elem[0], n_mid01, n_mid20],
                [n_mid01, elem[1], n_mid12],
                [n_mid20, n_mid12, elem[2]],
                [n_mid01, n_mid12, n_mid20]
            ])
        else:
            # Keep original element
            refined_elements.append(elem.tolist())
    
    return np.array(refined_nodes), np.array(refined_elements)

# Start with coarse mesh
nodes_coarse, elem_coarse = create_structured_mesh(4, 4)

# Apply adaptive refinement
nodes_refined, elem_refined = adaptive_refinement(nodes_coarse, elem_coarse)

print(f"Original mesh: {len(nodes_coarse)} nodes, {len(elem_coarse)} elements")
print(f"Refined mesh: {len(nodes_refined)} nodes, {len(elem_refined)} elements")

In [ ]:
# Visualize refinement
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Original mesh
ax1.triplot(nodes_coarse[:, 0], nodes_coarse[:, 1], elem_coarse, 'b-', linewidth=0.5)
ax1.plot(nodes_coarse[:, 0], nodes_coarse[:, 1], 'ro', markersize=4)
ax1.set_title('Original Coarse Mesh')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Refined mesh
ax2.triplot(nodes_refined[:, 0], nodes_refined[:, 1], elem_refined, 'b-', linewidth=0.5)
ax2.plot(nodes_refined[:, 0], nodes_refined[:, 1], 'ro', markersize=4)
ax2.set_title('Adaptively Refined Mesh')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

# Highlight refined region
circle = plt.Circle((0.5, 0.5), 0.3, fill=False, 
                    edgecolor='red', linewidth=2, linestyle='--')
ax2.add_patch(circle)

plt.tight_layout()
plt.show()

print("Adaptive Refinement Benefits:")
print("- Concentrates computational effort where needed")
print("- Reduces total DOFs compared to uniform refinement")
print("- Improves accuracy in critical regions")

## 5. Practical Guidelines

### Best Practices for Mesh Generation

1. **Element Quality**: Keep aspect ratios < 5, minimum angles > 30°
2. **Mesh Density**: Use finer meshes where solution gradients are high
3. **Transition**: Ensure smooth transition between element sizes
4. **Boundary Alignment**: Align mesh edges with important geometric features

### Common Pitfalls to Avoid

1. **Skinny triangles**: Cause numerical instability
2. **Abrupt size changes**: Lead to local errors
3. **Poor boundary representation**: Inaccurate geometry
4. **Over-refinement**: Wasted computational resources

In [ ]:
# Summary statistics for our meshes
print("=== MESH QUALITY SUMMARY ===")
print(f"\nStructured Mesh:")
print(f"  Elements: {len(elem_struct)}")
print(f"  Aspect ratio: {np.mean(ar_struct):.2f} ± {np.std(ar_struct):.2f}")
print(f"  Min angle: {np.mean(min_ang_struct):.1f}° ± {np.std(min_ang_struct):.1f}°")

print(f"\nUnstructured Mesh:")
print(f"  Elements: {len(elem_unstruct)}")
print(f"  Aspect ratio: {np.mean(ar_unstruct):.2f} ± {np.std(ar_unstruct):.2f}")
print(f"  Min angle: {np.mean(min_ang_unstruct):.1f}° ± {np.std(min_ang_unstruct):.1f}°")

print(f"\nRefined Mesh:")
ar_refined, min_ang_refined = triangle_quality(nodes_refined, elem_refined)
print(f"  Elements: {len(elem_refined)}")
print(f"  Aspect ratio: {np.mean(ar_refined):.2f} ± {np.std(ar_refined):.2f}")
print(f"  Min angle: {np.mean(min_ang_refined):.1f}° ± {np.std(min_ang_refined):.1f}°")

print("\n=== RECOMMENDATIONS ===")
print("1. Use structured meshes for simple geometries")
print("2. Use Delaunay triangulation for unstructured meshes")
print("3. Always check mesh quality before solving")
print("4. Apply adaptive refinement for complex solutions")
print("5. Balance accuracy requirements with computational cost")